# Pipeline de Treinamento DQN para QA com NLPGym
Este notebook mostra passo a passo como preparar, treinar e avaliar um agente DQN para tarefas de Question Answering usando a biblioteca **nlp-gym**.
**Atualização:** adicionamos uma amostra antes de chamar `check_env` para evitar erro de ambiente vazio.

## 1. Instalar dependências
```bash
pip install -r requirements.txt
pip install stable-baselines3 torch scikit-learn nbformat
```

## 2. Imports e verificação do ambiente
Wrapper de compatibilidade para Gymnasium com adição de um sample antes de `reset`.

In [ ]:
import gymnasium as gym
from gymnasium import spaces as new_spaces
import gym as old_gym
from stable_baselines3.common.env_checker import check_env
from nlp_gym.data_pools.custom_question_answering_pools import QASC
from nlp_gym.envs.question_answering.env import QAEnv
from nlp_gym.envs.question_answering.featurizer import InformedFeaturizer

class EnvCompatibility(gym.Env):
    def __init__(self, legacy_env):
        super().__init__()
        self.legacy_env = legacy_env
        self.action_space = self._convert_space(legacy_env.action_space)
        self.observation_space = self._convert_space(legacy_env.observation_space)

    def _convert_space(self, space):
        if isinstance(space, old_gym.spaces.Box):
            return new_spaces.Box(low=space.low, high=space.high, dtype=space.dtype)
        if isinstance(space, old_gym.spaces.Discrete):
            return new_spaces.Discrete(space.n)
        if isinstance(space, old_gym.spaces.MultiBinary):
            return new_spaces.MultiBinary(space.n)
        if isinstance(space, old_gym.spaces.MultiDiscrete):
            return new_spaces.MultiDiscrete(space.nvec)
        if isinstance(space, old_gym.spaces.Tuple):
            return new_spaces.Tuple(tuple(self._convert_space(s) for s in space.spaces))
        if isinstance(space, old_gym.spaces.Dict):
            return new_spaces.Dict({k: self._convert_space(v) for k,v in space.spaces.items()})
        raise ValueError(f"Tipo de espaço não suportado: {type(space)}")

    def reset(self, *, seed=None, **kwargs):
        _ = seed
        obs = self.legacy_env.reset()
        return obs, {}

    def step(self, action):
        obs, reward, done, info = self.legacy_env.step(action)
        return obs, reward, done, False, info

    def render(self, *args, **kwargs):
        return self.legacy_env.render(*args, **kwargs)

    def close(self):
        return self.legacy_env.close()

# Preparar ambiente legado
featurizer = InformedFeaturizer()
raw_env = QAEnv(observation_featurizer=featurizer)

# Adicionar pelo menos um sample antes de reset/check
pool = QASC.prepare('train')
sample, weight = next(iter(pool))
raw_env.add_sample(sample, weight)

# Envolver em compatibilidade e checar
compat_env = EnvCompatibility(raw_env)
check_env(compat_env, warn=True)

## 3. Preparar dados e vetorização

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from nlp_gym.data_pools.custom_question_answering_pools import QASC

train_pool = QASC.prepare('train')
featurizer = InformedFeaturizer()
base_env = QAEnv(observation_featurizer=featurizer)
for sample, weight in train_pool:
    base_env.add_sample(sample, weight)
env = Monitor(base_env)
vec_env = DummyVecEnv([lambda: env])